# IMPORTS:

In [14]:
import warnings
warnings.filterwarnings('ignore')
import pandas as pd
from sklearn.model_selection import train_test_split
import optuna
from optuna.pruners import MedianPruner
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import RFE

In [2]:
df = pd.read_csv("cardio_data_processed.csv")

# Preprocessing & Feature Engineering:

In [3]:
df.drop('id', axis=1, inplace = True) # Dropped id becoz its useless for prediction...
df["age"] = (df.age/365).astype(int) # Converting Age in years as its given in days...
df.drop('age_years', axis = 1, inplace = True) # Dropped id becoz its useless for prediction...
df.drop('bp_category_encoded', axis = 1, inplace = True) # Dropped id becoz its useless for prediction...
df.drop('bp_category', axis = 1, inplace = True) # Dropped id becoz its useless for prediction...

In [4]:
df["pulse_pressure"] = df["ap_hi"] - df["ap_lo"]
df["mean_arterial_pressure"] = df["ap_hi"] + 2*df["ap_lo"]/3

In [5]:
#Checking if there any missing or false value & dropping that...
df[['ap_hi', 'ap_lo', 'pulse_pressure', 'mean_arterial_pressure']].describe()
print((df['pulse_pressure'] < 0).sum()) #Since i got 3 value whose pulse_pressure is < 0 which make no sense as there is an error, so i gonna drop this 3 rows
df = df[df['pulse_pressure'] >= 0] #It removed those 3rows but index still show 0 to 68204 so we need to reset index...
df = df.reset_index(drop=True) #index reset...

3


# Train Test Split:

In [6]:
X = df.drop('cardio', axis = 1)
y = df['cardio']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 10)

# Optuna:

In [13]:
def objective(trial):
    """
    Objective function for Optuna to optimize Decision Tree hyperparameters
    """
    
    # Define hyperparameter search space
    max_depth = trial.suggest_int('max_depth', 3, 20)
    min_samples_split = trial.suggest_int('min_samples_split', 2, 50)
    min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 30)
    criterion = trial.suggest_categorical('criterion', ['gini', 'entropy'])
    class_weight = trial.suggest_categorical('class_weight', ['balanced', None])
    
    # Create pipeline with suggested hyperparameters
    pipeline = Pipeline([
        ("scaler", StandardScaler()),
        ("model", DecisionTreeClassifier(
            random_state=10,
            max_depth=max_depth,
            min_samples_split=min_samples_split,
            min_samples_leaf=min_samples_leaf,
            criterion=criterion,
            class_weight=class_weight
        ))
    ])
    
    # Use Stratified 5-Fold CV to evaluate
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=10)
    
    # Define scoring metrics
    scoring = {
        'accuracy': 'accuracy',
        'precision': 'precision_weighted',
        'recall': 'recall_weighted',
        'f1': 'f1_weighted',
        'roc_auc': 'roc_auc_ovr_weighted'
    }
    
    # Perform cross-validation
    cv_results = cross_validate(
        pipeline, 
        X_train, 
        y_train, 
        cv=cv, 
        scoring=scoring,
        return_train_score=False
    )
    
    # Return mean F1 score for optimization
    mean_accuracy = cv_results['test_accuracy'].mean()
    
    return mean_accuracy

# Create study object
study = optuna.create_study(
    direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=10),
    pruner=MedianPruner()
)

# Run optimization
print("Starting Optuna optimization...")
print("=" * 70)

study.optimize(objective, n_trials=100, show_progress_bar=True)

# Get best trial
best_trial = study.best_trial

print("\n" + "=" * 70)
print("OPTUNA TUNING RESULTS")
print("=" * 70)
print(f"\nBest F1 Score: {best_trial.value:.4f}")
print(f"Number of Trials: {len(study.trials)}")
print(f"\nBest Hyperparameters:")
print("-" * 70)
for key, value in best_trial.params.items():
    print(f"  {key}: {value}")

print("=" * 70)

[I 2026-05-06 00:08:19,625] A new study created in memory with name: no-name-5d7042a8-3f36-4ac5-aa29-e57273dc77c5


Starting Optuna optimization...


  0%|          | 0/100 [00:00<?, ?it/s]

[I 2026-05-06 00:08:22,408] Trial 0 finished with value: 0.7123951770104959 and parameters: {'max_depth': 16, 'min_samples_split': 3, 'min_samples_leaf': 20, 'criterion': 'gini', 'class_weight': 'balanced'}. Best is trial 0 with value: 0.7123951770104959.
[I 2026-05-06 00:08:29,514] Trial 1 finished with value: 0.6996571272310217 and parameters: {'max_depth': 16, 'min_samples_split': 10, 'min_samples_leaf': 3, 'criterion': 'entropy', 'class_weight': None}. Best is trial 0 with value: 0.7123951770104959.
[I 2026-05-06 00:08:36,359] Trial 2 finished with value: 0.712248569598582 and parameters: {'max_depth': 17, 'min_samples_split': 32, 'min_samples_leaf': 22, 'criterion': 'entropy', 'class_weight': 'balanced'}. Best is trial 0 with value: 0.7123951770104959.
[I 2026-05-06 00:08:38,587] Trial 3 finished with value: 0.7271675599845541 and parameters: {'max_depth': 5, 'min_samples_split': 20, 'min_samples_leaf': 21, 'criterion': 'gini', 'class_weight': 'balanced'}. Best is trial 3 with val

# RFE feature selection loop:

In [20]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.feature_selection import RFE
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# RFE FEATURE SELECTION WITH RFE EMBEDDED IN CV (NO DATA LEAKAGE)
# ============================================================

# Using best hyperparameters from Optuna
best_params = best_trial.params

print("Starting RFE Feature Selection (with RFE embedded in CV)...")
print("=" * 80)
print(f"Using best hyperparameters from Optuna:")
for key, value in best_params.items():
    print(f"  {key}: {value}")
print("=" * 80)

# Feature count range to test (8-14 features)
feature_counts = range(8, 15)  # 8, 9, 10, 11, 12, 13, 14

# Dictionary to store results
rfe_results = {}

# Stratified 5-Fold CV
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=10)

# Loop through each feature count
for n_features in feature_counts:
    print(f"\nTesting with {n_features} features...")
    
    # Store metrics for all folds
    fold_accuracies = []
    fold_precisions = []
    fold_recalls = []
    fold_f1s = []
    fold_roc_aucs = []
    
    # CV Loop - RFE EMBEDDED INSIDE
    fold_idx = 0
    for train_idx, val_idx in cv.split(X_train, y_train):
        fold_idx += 1
        
        # Split data into train and validation
        X_fold_train, X_fold_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_fold_train, y_fold_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
        
        # Create pipeline with RFE
        pipeline = Pipeline([
            ("scaler", StandardScaler()),
            ("rfe", RFE(
                estimator=DecisionTreeClassifier(
                    random_state=10,
                    max_depth=best_params['max_depth'],
                    min_samples_split=best_params['min_samples_split'],
                    min_samples_leaf=best_params['min_samples_leaf'],
                    criterion=best_params['criterion'],
                    class_weight=best_params['class_weight']
                ),
                n_features_to_select=n_features,
                step=1
            )),
            ("model", DecisionTreeClassifier(
                random_state=10,
                max_depth=best_params['max_depth'],
                min_samples_split=best_params['min_samples_split'],
                min_samples_leaf=best_params['min_samples_leaf'],
                criterion=best_params['criterion'],
                class_weight=best_params['class_weight']
            ))
        ])
        
        # Fit on TRAINING fold (RFE learns feature importance from training fold only)
        pipeline.fit(X_fold_train, y_fold_train)
        
        # Predict on VALIDATION fold
        y_pred = pipeline.predict(X_fold_val)
        y_pred_proba = pipeline.predict_proba(X_fold_val)[:, 1]
        
        # Calculate metrics
        accuracy = accuracy_score(y_fold_val, y_pred)
        precision = precision_score(y_fold_val, y_pred, average='weighted')
        recall = recall_score(y_fold_val, y_pred, average='weighted')
        f1 = f1_score(y_fold_val, y_pred, average='weighted')
        roc_auc = roc_auc_score(y_fold_val, y_pred_proba, average='weighted')
        
        # Store results
        fold_accuracies.append(accuracy)
        fold_precisions.append(precision)
        fold_recalls.append(recall)
        fold_f1s.append(f1)
        fold_roc_aucs.append(roc_auc)
    
    # Store mean and std for this feature count
    rfe_results[n_features] = {
        'accuracy': np.array(fold_accuracies),
        'precision': np.array(fold_precisions),
        'recall': np.array(fold_recalls),
        'f1': np.array(fold_f1s),
        'roc_auc': np.array(fold_roc_aucs)
    }
    
    # Display results for this feature count
    mean_accuracy = np.mean(fold_accuracies)
    std_accuracy = np.std(fold_accuracies)
    print(f"  Accuracy: {mean_accuracy:.4f} ± {std_accuracy:.4f}")

# ============================================================
# SELECT BEST FEATURE COUNT (based on Accuracy)
# ============================================================

print("\n" + "=" * 80)
print("RFE RESULTS SUMMARY (with RFE embedded in CV)")
print("=" * 80)

# Create results dataframe
results_data = {}
for n_features in feature_counts:
    results_data[n_features] = {
        'Accuracy': f"{rfe_results[n_features]['accuracy'].mean():.3f} ± {rfe_results[n_features]['accuracy'].std():.3f}",
        'Precision': f"{rfe_results[n_features]['precision'].mean():.3f} ± {rfe_results[n_features]['precision'].std():.3f}",
        'Recall': f"{rfe_results[n_features]['recall'].mean():.3f} ± {rfe_results[n_features]['recall'].std():.3f}",
        'F1 Score': f"{rfe_results[n_features]['f1'].mean():.3f} ± {rfe_results[n_features]['f1'].std():.3f}",
        'ROC AUC': f"{rfe_results[n_features]['roc_auc'].mean():.3f} ± {rfe_results[n_features]['roc_auc'].std():.3f}"
    }

results_df = pd.DataFrame(results_data).T
print("\n")
print(results_df)

# Find best feature count based on Accuracy (mean across all folds)
best_feature_count = max(
    rfe_results.keys(),
    key=lambda x: rfe_results[x]['accuracy'].mean()
)

best_accuracy = rfe_results[best_feature_count]['accuracy'].mean()
best_accuracy_std = rfe_results[best_feature_count]['accuracy'].std()

print("\n" + "=" * 80)
print("BEST FEATURE COUNT")
print("=" * 80)
print(f"\nBest Number of Features: {best_feature_count}")
print(f"Best Accuracy: {best_accuracy:.4f} ± {best_accuracy_std:.4f}")
print("\nAll Metrics for Best Feature Count:")
print("-" * 80)
for metric_name in ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']:
    metric_values = rfe_results[best_feature_count][metric_name]
    mean = metric_values.mean()
    std = metric_values.std()
    print(f"  {metric_name.upper():10s}: {mean:.4f} ± {std:.4f}")

print("=" * 80)

# Store best feature count for next step
best_rfe_features = best_feature_count
print(f"\n✓ Best feature count saved: {best_rfe_features}")

Starting RFE Feature Selection (with RFE embedded in CV)...
Using best hyperparameters from Optuna:
  max_depth: 7
  min_samples_split: 36
  min_samples_leaf: 29
  criterion: entropy
  class_weight: balanced

Testing with 8 features...
  Accuracy: 0.7279 ± 0.0038

Testing with 9 features...
  Accuracy: 0.7278 ± 0.0046

Testing with 10 features...
  Accuracy: 0.7279 ± 0.0045

Testing with 11 features...
  Accuracy: 0.7278 ± 0.0044

Testing with 12 features...
  Accuracy: 0.7283 ± 0.0040

Testing with 13 features...
  Accuracy: 0.7282 ± 0.0040

Testing with 14 features...
  Accuracy: 0.7285 ± 0.0045

RFE RESULTS SUMMARY (with RFE embedded in CV)


         Accuracy      Precision         Recall       F1 Score        ROC AUC
8   0.728 ± 0.004  0.728 ± 0.004  0.728 ± 0.004  0.728 ± 0.004  0.790 ± 0.006
9   0.728 ± 0.005  0.728 ± 0.004  0.728 ± 0.005  0.728 ± 0.005  0.790 ± 0.007
10  0.728 ± 0.004  0.728 ± 0.004  0.728 ± 0.004  0.728 ± 0.005  0.790 ± 0.007
11  0.728 ± 0.004  0.728 ± 0.004  

# FINAL DT PIPELINE WITH RESULTS:

In [28]:
from sklearn.feature_selection import RFE
from sklearn.tree import DecisionTreeClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
import numpy as np

# ============================================================
# FINAL PIPELINE TEST: RFE → DECISION TREE (with RFE in CV)
# ============================================================

print("=" * 80)
print("FINAL PIPELINE TESTING (RFE embedded in CV)")
print("=" * 80)
print(f"\nConfiguration:")
print(f"  Best Feature Count: {best_rfe_features}")
print(f"  RFE Estimator: Decision Tree")
print(f"  Max Depth: {best_params['max_depth']}")
print(f"  Min Samples Split: {best_params['min_samples_split']}")
print(f"  Min Samples Leaf: {best_params['min_samples_leaf']}")
print(f"  Criterion: {best_params['criterion']}")
print(f"  Class Weight: {best_params['class_weight']}")
print("=" * 80)

# Stratified 5-Fold CV
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=10)

# Store metrics for all folds
fold_accuracies = []
fold_precisions = []
fold_recalls = []
fold_f1s = []
fold_roc_aucs = []

print("\nRunning 5-Fold Stratified Cross-Validation...")
print("-" * 80)
print(f"{'Fold':<6} {'Accuracy':<12} {'Precision':<12} {'Recall':<12} {'F1 Score':<12} {'ROC AUC':<12}")
print("-" * 80)

# CV Loop - RFE EMBEDDED INSIDE
fold_idx = 0
for train_idx, val_idx in cv.split(X_train, y_train):
    fold_idx += 1
    
    # Split data into train and validation
    X_fold_train, X_fold_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
    y_fold_train, y_fold_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
    
    # Create pipeline with RFE and best feature count
    pipeline = Pipeline([
        ("scaler", StandardScaler()),
        ("rfe", RFE(
            estimator=DecisionTreeClassifier(
                random_state=10,
                max_depth=best_params['max_depth'],
                min_samples_split=best_params['min_samples_split'],
                min_samples_leaf=best_params['min_samples_leaf'],
                criterion=best_params['criterion'],
                class_weight=best_params['class_weight']
            ),
            n_features_to_select=best_rfe_features,
            step=1
        )),
        ("model", DecisionTreeClassifier(
            random_state=10,
            max_depth=best_params['max_depth'],
            min_samples_split=best_params['min_samples_split'],
            min_samples_leaf=best_params['min_samples_leaf'],
            criterion=best_params['criterion'],
            class_weight=best_params['class_weight']
        ))
    ])
    
    # Fit on TRAINING fold (RFE learns from training fold only)
    pipeline.fit(X_fold_train, y_fold_train)
    
    # Predict on VALIDATION fold
    y_pred = pipeline.predict(X_fold_val)
    y_pred_proba = pipeline.predict_proba(X_fold_val)[:, 1]
    
    # Calculate metrics
    accuracy = accuracy_score(y_fold_val, y_pred)
    precision = precision_score(y_fold_val, y_pred, average='weighted')
    recall = recall_score(y_fold_val, y_pred, average='weighted')
    f1 = f1_score(y_fold_val, y_pred, average='weighted')
    roc_auc = roc_auc_score(y_fold_val, y_pred_proba, average='weighted')
    
    # Store results
    fold_accuracies.append(accuracy)
    fold_precisions.append(precision)
    fold_recalls.append(recall)
    fold_f1s.append(f1)
    fold_roc_aucs.append(roc_auc)
    
    # Print fold results
    print(f"{fold_idx:<6} {accuracy:<12.4f} {precision:<12.4f} {recall:<12.4f} {f1:<12.4f} {roc_auc:<12.4f}")

# Calculate and display summary
print("\n" + "=" * 80)
print("FINAL PIPELINE RESULTS (Mean ± Std)")
print("=" * 80)

print("\n")
print(f"{'Accuracy':<12s}: {np.mean(fold_accuracies):.3f} ± {np.std(fold_accuracies):.3f}")
print(f"{'Precision':<12s}: {np.mean(fold_precisions):.3f} ± {np.std(fold_precisions):.3f}")
print(f"{'Recall':<12s}: {np.mean(fold_recalls):.3f} ± {np.std(fold_recalls):.3f}")
print(f"{'F1 Score':<12s}: {np.mean(fold_f1s):.3f} ± {np.std(fold_f1s):.3f}")
print(f"{'ROC AUC':<12s}: {np.mean(fold_roc_aucs):.3f} ± {np.std(fold_roc_aucs):.3f}")

print("\n" + "=" * 80)

FINAL PIPELINE TESTING (RFE embedded in CV)

Configuration:
  Best Feature Count: 14
  RFE Estimator: Decision Tree
  Max Depth: 7
  Min Samples Split: 36
  Min Samples Leaf: 29
  Criterion: entropy
  Class Weight: balanced

Running 5-Fold Stratified Cross-Validation...
--------------------------------------------------------------------------------
Fold   Accuracy     Precision    Recall       F1 Score     ROC AUC     
--------------------------------------------------------------------------------
1      0.7370       0.7371       0.7370       0.7369       0.8030      
2      0.7244       0.7250       0.7244       0.7241       0.7834      
3      0.7286       0.7305       0.7286       0.7278       0.7910      
4      0.7251       0.7261       0.7251       0.7245       0.7862      
5      0.7274       0.7281       0.7274       0.7269       0.7870      

FINAL PIPELINE RESULTS (Mean ± Std)


Accuracy    : 0.729 ± 0.005
Precision   : 0.729 ± 0.004
Recall      : 0.729 ± 0.005
F1 Score    